# Импорт библиотек:

In [1]:
import serial
import serial.tools.list_ports
import time
import numpy as np
import plotly
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import pandas as pd
from IPython.display import display, clear_output
from ipywidgets import Button, Output

# Получаем список всех доступных портов:

In [2]:
ports = serial.tools.list_ports.comports()

for port in ports:
    print(port.device, port.manufacturer, port.description)

COM12 FTDI USB Serial Port (COM12)


# Читаем данные с СОМ-порта:

In [3]:
def decode_packet(packet):
    if len(packet) != 5:
        return "Invalid packet. Packet length does not equals '5'."
    sync, msb, mid, lsb, checksum = packet

    if sync not in [0xA0, 0xA1, 0xA2, 0xA3]:
        return "Invalid packet. Sync byte error."

    if checksum != (msb ^ mid ^ lsb):
        return "Invalid packet. Checksum error."

    value = (msb << 16) | (mid << 8) | lsb    # Combine three bytes to one 24-bit number
    channel = sync - 0xA0 + 1    # Convert sync byte to number of channel

    return channel, value

In [4]:
ports = serial.tools.list_ports.comports()

for port in ports:
    print(port.device, port.manufacturer, port.description)

COM12 FTDI USB Serial Port (COM12)


In [5]:
def decode_packet(packet):
    """Декодирование одного пакета"""
    if len(packet) != 5:
        return None
    
    sync, msb, mid, lsb, checksum = packet
    
    # Проверка синхробайта
    if sync not in [0xA0, 0xA1, 0xA2, 0xA3]:
        return None
    
    # Проверка контрольной суммы
    if checksum != (msb ^ mid ^ lsb):
        return None
    
    # Декодирование значения
    value = (msb << 16) | (mid << 8) | lsb
    channel = sync - 0xA0 + 1
    
    return channel, value

# Параметры
port = "COM12"
baudrate = 115200
dt = 0.01  # интервал обновления
max_points = 1000  # максимальное количество точек на графике

# Создание графиков
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=('Канал 1', 'Канал 2', 
                    'Канал 3', 'Канал 4'),
    vertical_spacing=0.1
)

# Добавление трасс
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=1, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=2, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=3, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=4, col=1)

fig.update_layout(
    height=1200,
    showlegend=False,
    title_text="Данные с последовательного порта"
)

# Настройка осей
#for i in range(1, 5):
#    fig.update_xaxes(title_text="Время (с)", range=[0, max_points * dt], row=i, col=1)
#    fig.update_yaxes(title_text="Значение", range=[0, 100_000_000], row=i, col=1)


for i in range(1, 5):
    fig.update_xaxes(title_text="Время (с)", range=[0, max_points * dt], row=i, col=1)
    fig.update_yaxes(title_text="Значение", row=i, col=1)


# Создание виджета
fig_widget = go.FigureWidget(fig)
display(fig_widget)

print(f"Connecting to port: {port}...")
try:
    ser = serial.Serial(port, baudrate, timeout=1)
    print("Connected!")
except serial.SerialException as e:
    print(f"Error connecting to {port}: {e}")
    exit()

buffer = bytearray()

# Хранение данных для графиков
data_lists = [[], [], [], []]  # список списков для каждого канала
time_list = []  # общий список времени

t = 0.0  # начальное время

try:
    while True:
        # Чтение данных из порта
        if ser.in_waiting:
            raw_data = ser.read(ser.in_waiting)
            buffer.extend(raw_data)
        
        # Обработка буфера
        i = 0
        packets_processed = 0
        
        while i <= len(buffer) - 5:
            if buffer[i] in [0xA0, 0xA1, 0xA2, 0xA3]:
                packet = buffer[i:i+5]
                result = decode_packet(packet)
                if result:
                    channel, value = result
                    #print(f"Channel {channel}: {value}")
                    
                    # Добавляем значение в соответствующий список
                    channel_idx = channel - 1
                    data_lists[channel_idx].append(value)
                    
                    # Ограничиваем размер списка
                    if len(data_lists[channel_idx]) > max_points:
                        data_lists[channel_idx] = data_lists[channel_idx][-max_points:]
                    
                    # Обновляем время
                    time_list.append(t)
                    if len(time_list) > max_points:
                        time_list = time_list[-max_points:]
                    
                    t += dt
                    packets_processed += 1
                    i += 5
                    continue
            i += 1
        
        # Удаляем обработанные данные из буфера
        if i > 0:
            buffer = buffer[i:]
        
        # ОБНОВЛЕНИЕ ГРАФИКОВ - КЛЮЧЕВОЙ МОМЕНТ
        # Создаем новые данные для каждого графика
        for ch in range(4):
            if data_lists[ch]:  # Если есть данные
                # Создаем список времени для этого канала
                #if ch == 0:
                    # Для первого канала используем общий time_list
                    #ch_time = time_list[-len(data_lists[ch]):] if time_list else []
                #else:
                    # Для остальных каналов создаем время на основе индексов
                    # ch_time = list(range(len(data_lists[ch])))  # Просто номера точек
                    # Или используем время с учетом dt
                ch_time = [i * 0.01 for i in range(len(data_lists[ch]))]
                
                # Обновляем график
                fig_widget.data[ch].x = ch_time
                fig_widget.data[ch].y = data_lists[ch]
        
        # Принудительное обновление виджета (не всегда нужно, но помогает)
        #fig_widget.update_layout(yaxis=(0, max(max(data_lists[ch]))))
        
        # Небольшая пауза для снижения нагрузки CPU
        time.sleep(0.001)
        
except KeyboardInterrupt:
    print("\nStopping...")
except serial.SerialException as se:
    print(f"Serial port error: {se}")
except Exception as e:
    print(f"Error: {e}")
finally:
    ser.close()
    print("Port closed")

# Вывод статистики после остановки
print(f"\nStatistics:")
for i in range(4):
    print(f"Channel {i+1}: {len(data_lists[i])} points")

FigureWidget({
    'data': [{'mode': 'lines',
              'type': 'scatter',
              'uid': '6ec959e8-92bc-4a1c-a18c-3dbcde007a48',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'mode': 'lines',
              'type': 'scatter',
              'uid': 'a0bd9c49-e2d1-4b6b-97c8-e7bcc5ab9d32',
              'x': [],
              'xaxis': 'x2',
              'y': [],
              'yaxis': 'y2'},
             {'mode': 'lines',
              'type': 'scatter',
              'uid': '3a7951aa-1804-4391-8cc2-6e25656a4c9d',
              'x': [],
              'xaxis': 'x3',
              'y': [],
              'yaxis': 'y3'},
             {'mode': 'lines',
              'type': 'scatter',
              'uid': '449416d2-6930-4a73-9154-1b00bab22d32',
              'x': [],
              'xaxis': 'x4',
              'y': [],
              'yaxis': 'y4'}],
    'layout': {'annotations': [{'font': {'size': 16},
            

Connecting to port: COM12...
Connected!

Stopping...
Port closed

Statistics:
Channel 1: 1000 points
Channel 2: 1000 points
Channel 3: 1000 points
Channel 4: 1000 points


In [6]:
%pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [7]:
x = np.arange(0, 100)

def measure(x):
    yield [np.sin(0.2 * x), np.cos(0.4 * x)]

def f(x):
    return np.sin(x)

In [8]:
test_data = []
for i in range(100):
    test_data.append(f(i))
test_data

[np.float64(0.0),
 np.float64(0.8414709848078965),
 np.float64(0.9092974268256817),
 np.float64(0.1411200080598672),
 np.float64(-0.7568024953079282),
 np.float64(-0.9589242746631385),
 np.float64(-0.27941549819892586),
 np.float64(0.6569865987187891),
 np.float64(0.9893582466233818),
 np.float64(0.4121184852417566),
 np.float64(-0.5440211108893698),
 np.float64(-0.9999902065507035),
 np.float64(-0.5365729180004349),
 np.float64(0.4201670368266409),
 np.float64(0.9906073556948704),
 np.float64(0.6502878401571168),
 np.float64(-0.2879033166650653),
 np.float64(-0.9613974918795568),
 np.float64(-0.7509872467716762),
 np.float64(0.14987720966295234),
 np.float64(0.9129452507276277),
 np.float64(0.8366556385360561),
 np.float64(-0.008851309290403876),
 np.float64(-0.8462204041751706),
 np.float64(-0.9055783620066238),
 np.float64(-0.13235175009777303),
 np.float64(0.7625584504796027),
 np.float64(0.956375928404503),
 np.float64(0.27090578830786904),
 np.float64(-0.6636338842129675),
 np.fl

### Вариант от DeepSeek

In [ ]:
data = [np.random.randint(0,100) for _ in range(100)]

fig = go.FigureWidget(go.Scatter(
    x=[],
    y=[],
    mode='lines'
))

display(fig)

for i in range(100):
    data.pop(0)
    data.append(np.random.randint(0,100))

    fig.data[0].x = list(range(len(data)))
    fig.data[0].y = data

    time.sleep(0.1)

FigureWidget({
    'data': [{'mode': 'lines', 'type': 'scatter', 'uid': '29c230fd-984e-4542-8bf0-959ce6c875b8', 'x': [], 'y': []}],
    'layout': {'template': '...'}
})

KeyboardInterrupt: 

In [ ]:
# Инициализация данных для 4 графиков
data1 = [np.random.randint(0, 100) for _ in range(100)]
data2 = [np.random.randint(0, 100) for _ in range(100)]
data3 = [np.random.randint(0, 100) for _ in range(100)]
data4 = [np.random.randint(0, 100) for _ in range(100)]

# Создаем подграфики 2x2
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=('График 1', 'График 2', 'График 3', 'График 4'),
    vertical_spacing=0.1
)

# Добавляем трассы для каждого графика
fig.add_trace(go.Scatter(x=list(range(100)), y=data1, mode='lines', name='График 1'), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(100)), y=data2, mode='lines', name='График 2'), row=2, col=1)
fig.add_trace(go.Scatter(x=list(range(100)), y=data3, mode='lines', name='График 3'), row=3, col=1)
fig.add_trace(go.Scatter(x=list(range(100)), y=data4, mode='lines', name='График 4'), row=4, col=1)

# Настройка внешнего вида
fig.update_layout(
    title='4 графика в реальном времени',
    height=1200,
    showlegend=False,
    template='plotly_white'
)

# Настройка осей для каждого подграфика
fig.update_xaxes(title_text="Индекс", row=1, col=1)
fig.update_xaxes(title_text="Индекс", row=2, col=1)
fig.update_xaxes(title_text="Индекс", row=3, col=1)
fig.update_xaxes(title_text="Индекс", row=4, col=1)
fig.update_yaxes(title_text="Значение", row=1, col=1)
fig.update_yaxes(title_text="Значение", row=2, col=1)
fig.update_yaxes(title_text="Значение", row=3, col=1)
fig.update_yaxes(title_text="Значение", row=4, col=1)

# Устанавливаем диапазоны для всех осей
for i in range(1, 5):
    fig.update_xaxes(range=[0, 100], row=i, col=1)
    fig.update_yaxes(range=[0, 100], row=i, col=1)

# Преобразуем в FigureWidget для интерактивного обновления
fig = go.FigureWidget(fig)

# Отображаем график
display(fig)

# Обновляем данные в цикле
for i in range(100):
    # Обновляем данные
    data1.pop(0)
    data1.append(np.random.randint(0, 100))
    
    data2.pop(0)
    data2.append(np.random.randint(0, 100))
    
    data3.pop(0)
    data3.append(np.random.randint(0, 100))
    
    data4.pop(0)
    data4.append(np.random.randint(0, 100))
    
    # Обновляем графики
    with fig.batch_update():
        fig.data[0].x = list(range(len(data1)))
        fig.data[0].y = data1
        
        fig.data[1].x = list(range(len(data2)))
        fig.data[1].y = data2
        
        fig.data[2].x = list(range(len(data3)))
        fig.data[2].y = data3
        
        fig.data[3].x = list(range(len(data4)))
        fig.data[3].y = data4
        
        # Обновляем заголовок с номером итерации
        fig.layout.title.text = f'4 графика в реальном времени (обновление {i+1}/100)'
    
    time.sleep(0.1)

print("Обновление завершено!") 

In [ ]:
def sinus_1(x):
    return np.sin(x)

def sinus_2(x):
    return np.sin(2 * x / 3)

def sinus_3(x):
    return np.sin(2 * x)

def sinus_4(x):
    return np.sin(2 / (1 + x))

In [ ]:
data1, data2, data3, data4 = [0], [0], [0], [0]
dt = 0.1
t = 0

#while x := str(input("Waiting for command: ")).lower() != "q":
while True:
    x_time.append(t)
    data1.append(sinus_1(t))
    data2.append(sinus_2(t))
    data3.append(sinus_3(t))
    data4.append(sinus_4(t))
    t += dt
    
    if len(x_time) > 100:
        data1 = data1[1:]
        data2 = data2[1:]
        data3 = data3[1:]
        data4 = data4[1:]
    print(round(t, 2), "\t", data1[-1], "\t", data2[-1], "\t", data3[-1], "\t", data4[-1])
    time.sleep(dt)

In [ ]:
t = 0
x_time = []
dt = 0.1

fig = go.Figure()

fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=('График 1', 'График 2', 'График 3', 'График 4'),
    vertical_spacing=0.1
)

fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=1, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=2, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=3, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=4, col=1)

fig.update_layout(
#    title='4 графика',
    height=1200,
    showlegend=False,
)

for i in range(1, 5):
    fig.update_xaxes(range=[0, 10], row=i, col=1)
    fig.update_yaxes(range=[-2, 2], row=i, col=1)

fig = go.FigureWidget(fig)
display(fig)


while True:
    x_time.append(t)
    data1.append(sinus_1(t))
    data2.append(sinus_2(t))
    data3.append(sinus_3(t))
    data4.append(sinus_4(t))
    t += dt
    
    if len(x_time) > 100:
        data1 = data1[1:]
        data2 = data2[1:]
        data3 = data3[1:]
        data4 = data4[1:]

    with fig.batch_update():
        
        fig.data[0].x = x_time
        fig.data[0].y = data1
        
        fig.data[1].x = x_time
        fig.data[1].y = data2
        
        fig.data[2].x = x_time
        fig.data[2].y = data3
        
        fig.data[3].x = x_time
        fig.data[3].y = data4

    time.sleep(dt)
    

In [ ]:
while True:
    command = str(input("Waiting for command: "))
    if command.lower() == "q":
        break

In [ ]:
# Инициализация данных для 4 графиков
data1 = [np.random.randint(0, 100) for _ in range(100)]
data2 = [np.random.randint(0, 100) for _ in range(100)]
data3 = [np.random.randint(0, 100) for _ in range(100)]
data4 = [np.random.randint(0, 100) for _ in range(100)]

# Создаем подграфики 2x2
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=('График 1', 'График 2', 'График 3', 'График 4'),
    vertical_spacing=0.1
)

# Добавляем трассы для каждого графика
fig.add_trace(go.Scatter(x=list(range(100)), y=data1, mode='lines', name='График 1'), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(100)), y=data2, mode='lines', name='График 2'), row=2, col=1)
fig.add_trace(go.Scatter(x=list(range(100)), y=data3, mode='lines', name='График 3'), row=3, col=1)
fig.add_trace(go.Scatter(x=list(range(100)), y=data4, mode='lines', name='График 4'), row=4, col=1)

# Настройка внешнего вида
fig.update_layout(
    title='4 графика в реальном времени',
    height=1200,
    showlegend=False,
    template='plotly_white'
)

# Настройка осей для каждого подграфика
fig.update_xaxes(title_text="Индекс", row=1, col=1)
fig.update_xaxes(title_text="Индекс", row=2, col=1)
fig.update_xaxes(title_text="Индекс", row=3, col=1)
fig.update_xaxes(title_text="Индекс", row=4, col=1)
fig.update_yaxes(title_text="Значение", row=1, col=1)
fig.update_yaxes(title_text="Значение", row=2, col=1)
fig.update_yaxes(title_text="Значение", row=3, col=1)
fig.update_yaxes(title_text="Значение", row=4, col=1)

# Устанавливаем диапазоны для всех осей
for i in range(1, 5):
    fig.update_xaxes(range=[0, 100], row=i, col=1)
    fig.update_yaxes(range=[0, 100], row=i, col=1)

# Преобразуем в FigureWidget для интерактивного обновления
fig = go.FigureWidget(fig)

# Отображаем график
display(fig)

# Обновляем данные в цикле
for i in range(100):
    # Обновляем данные
    data1.pop(0)
    data1.append(np.random.randint(0, 100))
    
    data2.pop(0)
    data2.append(np.random.randint(0, 100))
    
    data3.pop(0)
    data3.append(np.random.randint(0, 100))
    
    data4.pop(0)
    data4.append(np.random.randint(0, 100))
    
    # Обновляем графики
    with fig.batch_update():
        fig.data[0].x = list(range(len(data1)))
        fig.data[0].y = data1
        
        fig.data[1].x = list(range(len(data2)))
        fig.data[1].y = data2
        
        fig.data[2].x = list(range(len(data3)))
        fig.data[2].y = data3
        
        fig.data[3].x = list(range(len(data4)))
        fig.data[3].y = data4
        
        # Обновляем заголовок с номером итерации
        fig.layout.title.text = f'4 графика в реальном времени (обновление {i+1}/100)'
    
    time.sleep(0.1)

print("Обновление завершено!") 

In [ ]:
from ipywidgets import Button, Output

In [ ]:
button1 = Button(description="Click!", 
                 button_style='success'
                )

out = Output()

def on_button_clicked(b):
    with out:
        out.clear_output()
        print("Click!!!")

button1.on_click(on_button_clicked)

display(button1, out)

In [ ]:
some_list = [
    [1,2,3,4],
    [2,3,4,2],
    [6,4,7,1]
]

len(some_list)

In [ ]:
some_list = some_list[1:]
some_list

In [ ]:
len(some_list[0])

In [ ]:
max(max(some_list))